In [1]:
from pathlib import Path
import json
import time
import gc

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Models directory: {MODELS_DIR}")

Project root: c:\Projects\creditlens-ai
Models directory: c:\Projects\creditlens-ai\models


In [2]:
feature_files = {
    "bureau": (
        INTERIM_DIR /
        "bureau_customer_features.csv"
    ),
    "previous": (
        INTERIM_DIR /
        "previous_application_customer_features.csv"
    ),
    "installments": (
        INTERIM_DIR /
        "installments_customer_features.csv"
    ),
}

application_train = pd.read_csv(
    DATA_DIR / "application_train.csv",
    low_memory=False
)

bureau_features = pd.read_csv(
    feature_files["bureau"]
)

previous_features = pd.read_csv(
    feature_files["previous"]
)

installment_features = pd.read_csv(
    feature_files["installments"]
)

print(f"Application:  {application_train.shape}")
print(f"Bureau:       {bureau_features.shape}")
print(f"Previous:     {previous_features.shape}")
print(f"Installments: {installment_features.shape}")

Application:  (307511, 122)
Bureau:       (305811, 20)
Previous:     (338857, 24)
Installments: (339587, 30)


In [3]:
train_full = (
    application_train
    .merge(
        bureau_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        previous_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        installment_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

engineered_flags = pd.DataFrame(
    {
        "BUREAU_HAS_HISTORY": (
            train_full["BUREAU_LOAN_COUNT"]
            .notna()
            .astype("int8")
        ),
        "PREV_HAS_HISTORY": (
            train_full["PREV_APPLICATION_COUNT"]
            .notna()
            .astype("int8")
        ),
        "INST_HAS_HISTORY": (
            train_full["INST_PAYMENT_RECORD_COUNT"]
            .notna()
            .astype("int8")
        ),
        "DAYS_EMPLOYED_ANOMALY": (
            train_full["DAYS_EMPLOYED"] == 365243
        ).astype("int8")
    },
    index=train_full.index
)

train_full = pd.concat(
    [train_full, engineered_flags],
    axis=1
)

train_full.loc[
    train_full["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(f"Final merged shape: {train_full.shape}")
print(
    "Duplicate customers:",
    train_full["SK_ID_CURR"].duplicated().sum()
)
print(
    "Missing TARGET:",
    train_full["TARGET"].isna().sum()
)

Final merged shape: (307511, 197)
Duplicate customers: 0
Missing TARGET: 0


In [4]:
y_final = train_full["TARGET"].copy()

X_final = train_full.drop(
    columns=[
        "TARGET",
        "SK_ID_CURR",
        "CODE_GENDER"
    ]
).copy()

categorical_columns_final = (
    X_final
    .select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

numeric_columns_final = (
    X_final
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

for column in categorical_columns_final:
    X_final[column] = (
        X_final[column]
        .astype("object")
        .where(
            X_final[column].notna(),
            "__MISSING__"
        )
        .astype(str)
    )

print(f"X final shape: {X_final.shape}")
print(f"y final shape: {y_final.shape}")

print(
    f"Numeric features: "
    f"{len(numeric_columns_final)}"
)

print(
    f"Categorical features: "
    f"{len(categorical_columns_final)}"
)

print(
    "CODE_GENDER in model:",
    "CODE_GENDER" in X_final.columns
)

print(
    "Categorical missing values:",
    X_final[categorical_columns_final]
    .isna()
    .sum()
    .sum()
)

X final shape: (307511, 194)
y final shape: (307511,)
Numeric features: 179
Categorical features: 15
CODE_GENDER in model: False
Categorical missing values: 0


In [5]:
final_feature_names = X_final.columns.tolist()

assert len(final_feature_names) == 194
assert len(set(final_feature_names)) == 194
assert "CODE_GENDER" not in final_feature_names
assert "TARGET" not in final_feature_names
assert "SK_ID_CURR" not in final_feature_names

print(f"Feature contract size: {len(final_feature_names)}")

print("\nFirst 10 features:")
for feature in final_feature_names[:10]:
    print(feature)

print("\nLast 10 features:")
for feature in final_feature_names[-10:]:
    print(feature)

Feature contract size: 194

First 10 features:
NAME_CONTRACT_TYPE
FLAG_OWN_CAR
FLAG_OWN_REALTY
CNT_CHILDREN
AMT_INCOME_TOTAL
AMT_CREDIT
AMT_ANNUITY
AMT_GOODS_PRICE
NAME_TYPE_SUITE
NAME_INCOME_TYPE

Last 10 features:
INST_UNDERPAID_COUNT
INST_LATE_PAYMENT_RATE
INST_LATE_7_RATE
INST_LATE_30_RATE
INST_UNDERPAID_RATE
INST_PAYMENT_TO_INSTALMENT_RATIO
BUREAU_HAS_HISTORY
PREV_HAS_HISTORY
INST_HAS_HISTORY
DAYS_EMPLOYED_ANOMALY


In [6]:
final_model = CatBoostClassifier(
    loss_function="Logloss",

    iterations=1470,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=5,

    auto_class_weights="Balanced",

    random_seed=42,
    thread_count=-1,

    verbose=250,
    allow_writing_files=False
)

final_model

CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=7, iterations=1470, l2_leaf_reg=5, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=250)

In [7]:
start_time = time.time()

final_model.fit(
    X_final,
    y_final,
    cat_features=categorical_columns_final
)

final_training_time = (
    time.time() - start_time
)

print(
    f"Final training time: "
    f"{final_training_time:.2f} seconds"
)

print(
    f"Final tree count: "
    f"{final_model.tree_count_}"
)

0:	learn: 0.6889771	total: 483ms	remaining: 11m 50s
250:	learn: 0.5694545	total: 56.3s	remaining: 4m 33s
500:	learn: 0.5505964	total: 1m 52s	remaining: 3m 37s
750:	learn: 0.5315989	total: 2m 50s	remaining: 2m 43s
1000:	learn: 0.5152407	total: 3m 48s	remaining: 1m 47s
1250:	learn: 0.5011280	total: 4m 45s	remaining: 50s
1469:	learn: 0.4895671	total: 5m 35s	remaining: 0us
Final training time: 337.23 seconds
Final tree count: 1470


In [8]:
final_model_path = (
    MODELS_DIR /
    "creditlens_catboost_gender_free.cbm"
)

final_model.save_model(
    str(final_model_path)
)

model_size_mb = (
    final_model_path.stat().st_size
    / (1024 ** 2)
)

print(f"Saved model: {final_model_path}")
print(f"Model size: {model_size_mb:.2f} MB")

Saved model: c:\Projects\creditlens-ai\models\creditlens_catboost_gender_free.cbm
Model size: 3.26 MB


In [9]:
feature_schema = {
    "feature_count": len(final_feature_names),

    "features": final_feature_names,

    "categorical_features": (
        categorical_columns_final
    ),

    "numeric_features": (
        numeric_columns_final
    ),

    "excluded_columns": [
        "TARGET",
        "SK_ID_CURR",
        "CODE_GENDER"
    ],

    "categorical_missing_value": (
        "__MISSING__"
    ),

    "days_employed_sentinel": 365243,

    "engineered_features": [
        "BUREAU_HAS_HISTORY",
        "PREV_HAS_HISTORY",
        "INST_HAS_HISTORY",
        "DAYS_EMPLOYED_ANOMALY"
    ]
}

feature_schema_path = (
    MODELS_DIR /
    "creditlens_feature_schema.json"
)

with open(
    feature_schema_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        feature_schema,
        file,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved schema: {feature_schema_path}")

Saved schema: c:\Projects\creditlens-ai\models\creditlens_feature_schema.json


In [10]:
model_metadata = {
    "model_name": (
        "CreditLens Gender-Free CatBoost"
    ),

    "model_type": (
        "CatBoostClassifier"
    ),

    "training_rows": int(
        len(X_final)
    ),

    "feature_count": int(
        X_final.shape[1]
    ),

    "categorical_feature_count": int(
        len(categorical_columns_final)
    ),

    "numeric_feature_count": int(
        len(numeric_columns_final)
    ),

    "iterations": 1470,
    "learning_rate": 0.03,
    "depth": 7,
    "l2_leaf_reg": 5,

    "class_weight_strategy": "Balanced",

    "random_seed": 42,

    "training_time_seconds": (
        final_training_time
    ),

    "excluded_sensitive_feature": (
        "CODE_GENDER"
    ),

    "development_validation": {
        "roc_auc": 0.782085,
        "pr_auc": 0.279699,
        "f1_at_0_50": 0.301294
    },

    "stability_cv_oof": {
        "roc_auc": 0.777891,
        "pr_auc": 0.268364,
        "f1_at_0_50": 0.300346,
        "recall_at_0_50": 0.639517
    },

    "probability_note": (
        "Model outputs are risk scores from a "
        "class-balanced model and should not be "
        "interpreted as calibrated probabilities "
        "of default without probability calibration."
    )
}

metadata_path = (
    MODELS_DIR /
    "creditlens_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved metadata: {metadata_path}")

Saved metadata: c:\Projects\creditlens-ai\models\creditlens_model_metadata.json


In [11]:
reloaded_model = CatBoostClassifier()

reloaded_model.load_model(
    str(final_model_path)
)

verification_sample = X_final.iloc[:100].copy()

original_predictions = (
    final_model.predict_proba(
        verification_sample
    )[:, 1]
)

reloaded_predictions = (
    reloaded_model.predict_proba(
        verification_sample
    )[:, 1]
)

max_prediction_difference = np.max(
    np.abs(
        original_predictions
        - reloaded_predictions
    )
)

print(
    f"Reloaded tree count: "
    f"{reloaded_model.tree_count_}"
)

print(
    f"Maximum prediction difference: "
    f"{max_prediction_difference:.12f}"
)

print(
    "Reload verification passed:",
    np.allclose(
        original_predictions,
        reloaded_predictions
    )
)

Reloaded tree count: 1470
Maximum prediction difference: 0.000000000000
Reload verification passed: True


In [12]:
application_test = pd.read_csv(
    DATA_DIR / "application_test.csv",
    low_memory=False
)

print(f"Application test shape: {application_test.shape}")
print(
    "TARGET in test:",
    "TARGET" in application_test.columns
)
print(
    "Duplicate customers:",
    application_test["SK_ID_CURR"]
    .duplicated()
    .sum()
)

Application test shape: (48744, 121)
TARGET in test: False
Duplicate customers: 0


In [13]:
test_full = (
    application_test
    .merge(
        bureau_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        previous_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        installment_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

print(f"Merged test shape: {test_full.shape}")
print(
    "Duplicate customers:",
    test_full["SK_ID_CURR"]
    .duplicated()
    .sum()
)

Merged test shape: (48744, 192)
Duplicate customers: 0


In [14]:
test_engineered_flags = pd.DataFrame(
    {
        "BUREAU_HAS_HISTORY": (
            test_full["BUREAU_LOAN_COUNT"]
            .notna()
            .astype("int8")
        ),
        "PREV_HAS_HISTORY": (
            test_full["PREV_APPLICATION_COUNT"]
            .notna()
            .astype("int8")
        ),
        "INST_HAS_HISTORY": (
            test_full["INST_PAYMENT_RECORD_COUNT"]
            .notna()
            .astype("int8")
        ),
        "DAYS_EMPLOYED_ANOMALY": (
            test_full["DAYS_EMPLOYED"] == 365243
        ).astype("int8")
    },
    index=test_full.index
)

test_full = pd.concat(
    [test_full, test_engineered_flags],
    axis=1
)

test_full.loc[
    test_full["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(
    f"Test shape after flags: "
    f"{test_full.shape}"
)

Test shape after flags: (48744, 196)


In [15]:
test_ids = test_full[
    "SK_ID_CURR"
].copy()

X_test_final = test_full.drop(
    columns=[
        "SK_ID_CURR",
        "CODE_GENDER"
    ]
).copy()

print(
    f"Test predictor shape: "
    f"{X_test_final.shape}"
)

print(
    "CODE_GENDER in test model input:",
    "CODE_GENDER" in X_test_final.columns
)

Test predictor shape: (48744, 194)
CODE_GENDER in test model input: False


In [16]:
missing_features = [
    feature
    for feature in final_feature_names
    if feature not in X_test_final.columns
]

extra_features = [
    feature
    for feature in X_test_final.columns
    if feature not in final_feature_names
]

print(
    f"Missing features: "
    f"{len(missing_features)}"
)

print(
    f"Extra features: "
    f"{len(extra_features)}"
)

print(
    "Exact feature set match:",
    set(X_test_final.columns)
    == set(final_feature_names)
)

X_test_final = X_test_final[
    final_feature_names
].copy()

print(
    "Exact feature order match:",
    X_test_final.columns.tolist()
    == final_feature_names
)

Missing features: 0
Extra features: 0
Exact feature set match: True
Exact feature order match: True


In [17]:
for column in categorical_columns_final:
    X_test_final[column] = (
        X_test_final[column]
        .astype("object")
        .where(
            X_test_final[column].notna(),
            "__MISSING__"
        )
        .astype(str)
    )

print(
    "Categorical missing values:",
    X_test_final[
        categorical_columns_final
    ]
    .isna()
    .sum()
    .sum()
)

print(
    f"Final inference shape: "
    f"{X_test_final.shape}"
)

Categorical missing values: 0
Final inference shape: (48744, 194)


In [18]:
start_time = time.time()

test_risk_scores = (
    reloaded_model.predict_proba(
        X_test_final
    )[:, 1]
)

inference_time = time.time() - start_time

print(
    f"Inference rows: "
    f"{len(test_risk_scores):,}"
)

print(
    f"Inference time: "
    f"{inference_time:.2f} seconds"
)

print(
    f"Minimum risk score: "
    f"{test_risk_scores.min():.6f}"
)

print(
    f"Mean risk score: "
    f"{test_risk_scores.mean():.6f}"
)

print(
    f"Maximum risk score: "
    f"{test_risk_scores.max():.6f}"
)

print(
    "NaN predictions:",
    np.isnan(test_risk_scores).sum()
)

Inference rows: 48,744
Inference time: 0.21 seconds
Minimum risk score: 0.006224
Mean risk score: 0.362170
Maximum risk score: 0.970388
NaN predictions: 0


In [19]:
test_predictions = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "risk_score": test_risk_scores
})

prediction_path = (
    REPORTS_DIR /
    "final_test_risk_scores.csv"
)

test_predictions.to_csv(
    prediction_path,
    index=False
)

print(f"Saved: {prediction_path}")

test_predictions.head(10)

Saved: c:\Projects\creditlens-ai\reports\final_test_risk_scores.csv


,SK_ID_CURR,risk_score
0,100001,0.346928
1,100005,0.627606
2,100013,0.189908
3,100028,0.344106
4,100038,0.641339
5,100042,0.383993
6,100057,0.098148
7,100065,0.168556
8,100066,0.106370
9,100067,0.338217


In [20]:
notebook_predictions = pd.read_csv(
    REPORTS_DIR / "final_test_risk_scores.csv"
)

cli_predictions = pd.read_csv(
    REPORTS_DIR / "inference_smoke_test.csv"
)

print(
    f"Notebook rows: "
    f"{len(notebook_predictions):,}"
)

print(
    f"CLI rows: "
    f"{len(cli_predictions):,}"
)

id_match = (
    notebook_predictions["SK_ID_CURR"]
    .equals(
        cli_predictions["SK_ID_CURR"]
    )
)

max_score_difference = np.max(
    np.abs(
        notebook_predictions["risk_score"].values
        - cli_predictions["risk_score"].values
    )
)

scores_match = np.allclose(
    notebook_predictions["risk_score"].values,
    cli_predictions["risk_score"].values,
    rtol=0,
    atol=1e-12
)

print(f"Customer IDs match: {id_match}")

print(
    f"Maximum score difference: "
    f"{max_score_difference:.12f}"
)

print(
    f"Risk scores match: "
    f"{scores_match}"
)

assert id_match
assert scores_match

print("\nInference regression check PASSED.")

Notebook rows: 48,744
CLI rows: 48,744
Customer IDs match: True
Maximum score difference: 0.000000000000
Risk scores match: True

Inference regression check PASSED.


In [21]:
with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:
    final_metadata = json.load(file)

final_metadata["inference_validation"] = {
    "application_test_rows": 48744,
    "feature_count": 194,
    "missing_features": 0,
    "extra_features": 0,
    "nan_predictions": 0,
    "reload_verification_passed": True,
    "cli_regression_test_passed": True
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

print(f"Updated metadata: {metadata_path}")

Updated metadata: c:\Projects\creditlens-ai\models\creditlens_model_metadata.json


## Final Model Artifact Validation

The selected gender-free CatBoost model was retrained on all 307,511 labeled training observations using 194 predictive features.

The final model was saved as a native CatBoost `.cbm` artifact together with a feature schema and model metadata.

An end-to-end inference test was performed on all 48,744 observations in `application_test.csv`.

Validation checks confirmed:

- 194 expected model features
- 0 missing features
- 0 unexpected features
- exact feature ordering
- 0 missing categorical values after preprocessing
- 0 NaN model outputs
- successful model save/reload verification
- identical predictions between notebook inference and `src/inference.py`

The model output is treated as a risk score rather than a calibrated probability of default because class balancing was used during model training.